# CBB Tournament — Notebook Exploration

Quick experiments on the College Basketball tournament dataset using the
kitchen platform notebook APIs. No CLI commands needed to produce tracked runs.

| Step | What | API |
|------|------|-----|
| 1 | Peek at data | `DataStore.preview()` |
| 2 | Try a quick inline model | `kitchen.experiment()` |
| 3 | Run a structured training pass | `kitchen.init_run()` |
| 4 | Compare runs | `mlflow.search_runs()` / `kitchen leaderboard` |

**Prerequisites:**
- `pip install -e ../kitchen-platform/kitchen -e ..` (dev) or `pip install rkoren-kitchen -e ..`
- CBB data in `data/raw/` — run `dvc pull` or place the Kaggle CSVs there
- Processed features built — run `kitchen run features` from `~/cbb-model/` first
- Start Jupyter from inside `~/cbb-model/notebooks/` (or `~/cbb-model/`)

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import mlflow
import pandas as pd
import yaml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

import kitchen
from kitchen.store import DataStore

# Project root — works whether Jupyter is started from notebooks/ or project root
project_root = next(
    d for d in [Path(".").resolve(), Path("..").resolve()] if (d / "params.yaml").exists()
)
store = DataStore(root=project_root)
params = yaml.safe_load((project_root / "params.yaml").read_text())
feature_candidates: list[str] = params["feature_candidates"]

# Pin the MLflow tracking URI to an absolute path so runs land in the correct
# database regardless of which directory Jupyter was started from.
tracking_uri = params.get("mlflow", {}).get("tracking_uri", "sqlite:///mlruns.db")
if tracking_uri.startswith("sqlite:///") and not tracking_uri.startswith("sqlite:////"):
    db_name = tracking_uri.removeprefix("sqlite:///")
    tracking_uri = f"sqlite:///{project_root / db_name}"
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri

print(f"Project root : {project_root}")
print(f"Experiment   : {params['experiment']}")
print(f"Tracking URI : {tracking_uri}")
print(f"Raw files    : {store.list('raw')[:4]} ...")
print(f"Processed    : {store.list('processed')[:6]} ...")

## Step 1 — Peek at data with `DataStore.preview()`

`store.preview(filename)` searches `data/processed/` first, then `data/raw/`,
and returns the first `n` rows (default 5). No path juggling needed.

In [ ]:
# Processed matchup dataset built by kitchen run features
store.preview("matchups.parquet")

In [ ]:
# Full shape + which feature_candidates are present
matchups = store.load_parquet("matchups.parquet")
features = [f for f in feature_candidates if f in matchups.columns]
print(f"Matchups : {matchups.shape[0]:,} rows  {matchups.shape[1]} cols")
print(f"Seasons  : {sorted(matchups['Season'].unique())}")
print(f"Features : {len(features)} of {len(feature_candidates)} candidates present")
print(f"Target   : Outcome  values={matchups['Outcome'].value_counts().to_dict()}")

## Step 2 — Quick idea with `kitchen.experiment()`

`kitchen.experiment()` is the zero-ceremony path:
- Write model code directly in a cell — no `Trainer` subclass required
- Auto-discovers `params.yaml` from the working directory or its parents
- Logs to the same MLflow experiment as `kitchen run train`
- Runs appear in `kitchen leaderboard` immediately after the cell runs

Metrics are logged as `loto_brier` (CBB's configured threshold metric) so they appear
in the default `kitchen leaderboard`. Here we use a single random split for speed;
production training uses full LOTO cross-validation via `kitchen run train`.

In [ ]:
matchups[features] = matchups[features].fillna(0)

# Exclude the 2026 holdout season (same convention as production)
holdout = params.get("evaluate", {}).get("holdout_season", 2026)
train_df = matchups[matchups["Season"] < holdout].copy()
X = train_df[features]
y = train_df["Outcome"]  # 1 = TeamA won

# Note: random split used for speed; production uses loto_cv (seasonal leave-one-out)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Idea A: logistic regression, C=1.0
with kitchen.experiment(
    params["experiment"],
    run_name="nb-logistic-C1",
    params={"model": "logistic", "C": 1.0, "n_features": len(features)},
) as run_a:
    lr_a = LogisticRegression(C=1.0, max_iter=2000)
    lr_a.fit(X_tr, y_tr)
    brier_a = brier_score_loss(y_val, lr_a.predict_proba(X_val)[:, 1])
    run_a.log(loto_brier=brier_a, val_brier=brier_a)
    run_a.set_tag("model_variant", "notebook-lr-C1")

print(f"run_id     : {run_a.run_id}")
print(f"loto_brier : {brier_a:.5f}")

In [ ]:
# Idea B: tighter regularisation — one parameter change, separate run
with kitchen.experiment(
    params["experiment"],
    run_name="nb-logistic-C01",
    params={"model": "logistic", "C": 0.1, "n_features": len(features)},
) as run_b:
    lr_b = LogisticRegression(C=0.1, max_iter=2000)
    lr_b.fit(X_tr, y_tr)
    brier_b = brier_score_loss(y_val, lr_b.predict_proba(X_val)[:, 1])
    run_b.log(loto_brier=brier_b, val_brier=brier_b)
    run_b.set_tag("model_variant", "notebook-lr-C01")

print(f"run_id     : {run_b.run_id}")
print(f"loto_brier : {brier_b:.5f}")

## Step 3 — Structured run with `kitchen.init_run()`

Once you have a `Trainer` subclass, `kitchen.init_run()` injects MLflow tracking
automatically — the same context that `kitchen run train` opens.

The lightweight trainer below uses logistic regression for speed so the notebook
runs end-to-end quickly. To run the full XGBoost/LOTO pipeline, replace
`LightCBBTrainer().run(...)` with:
```python
from src.train.run import train
train(params, store, tracker)
```
The result appears in the same `kitchen leaderboard` as `kitchen run train` output.

In [ ]:
from kitchen.steps import Trainer


class LightCBBTrainer(Trainer):
    """Lightweight CBB trainer for notebook iteration.

    Uses the matchup features but trains a logistic regression instead of the
    full XGBoost+LOTO pipeline. Same DataStore and params contract as production:
    DataStore loads matchups.parquet, params drives features and holdout season.
    """

    def fit(self, df: pd.DataFrame, params: dict) -> object:
        feat_candidates = params.get("feature_candidates", [])
        feats = [f for f in feat_candidates if f in df.columns]
        df = df.copy()
        df[feats] = df[feats].fillna(0)

        holdout_season = params.get("evaluate", {}).get("holdout_season", 2026)
        train_df = df[df["Season"] < holdout_season]
        X = train_df[feats]
        y = train_df["Outcome"]

        # Random split for speed — production uses loto_cv (seasonal leave-one-out)
        X_tr, X_vl, y_tr, y_vl = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LogisticRegression(C=0.5, max_iter=2000)
        model.fit(X_tr, y_tr)

        brier = brier_score_loss(y_vl, model.predict_proba(X_vl)[:, 1])
        mlflow.log_metric("loto_brier", brier)  # matches CBB threshold metric
        mlflow.log_metric("val_brier", brier)
        mlflow.log_param("n_features", len(feats))
        mlflow.set_tag("model_variant", "notebook-light")
        print(f"  loto_brier : {brier:.5f}  ({len(feats)} features)")
        return model

In [ ]:
# init_run() auto-discovers params.yaml and opens the same experiment as kitchen run train.
# Trainer.run() loads matchups.parquet from the DataStore and calls fit() above.
with kitchen.init_run(params, run_name="nb-init-run-light") as tracker:
    LightCBBTrainer().run(store, tracker, params)

print("Run logged — visible in `kitchen leaderboard`")

## Step 4 — Compare runs

All runs are tracked in the same experiment. Query them inline or use the CLI:

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[params["experiment"]],
    order_by=["metrics.loto_brier ASC"],
)

display_cols = ["run_id", "tags.mlflow.runName", "metrics.loto_brier", "tags.model_variant"]
display_cols += [c for c in runs.columns if c.startswith("params.")]
runs[display_cols].dropna(subset=["metrics.loto_brier"]).head(10)

### CLI equivalents

From a terminal in `~/cbb-model/`:

```bash
# Ranked leaderboard (loto_brier, lower=better)
kitchen leaderboard

# Add param columns
kitchen leaderboard --show-params model.max_depth,model.eta

# Diff two runs
kitchen diff <run_id_a> <run_id_b>

# Open MLflow UI
kitchen ui

# Run the full XGBoost+LOTO pipeline (takes several minutes)
kitchen run train --auto-promote
```